# DSPy RAG — Retrieval-Augmented Generation with Constraints

**Week 6 | Notebook 3 of 6**

**What you'll learn:**
- Setting up a retriever (ChromaDB / Qdrant)
- Building a basic RAG module
- Adding `dspy.Assert` for faithfulness
- Adding `dspy.Suggest` for length/format constraints
- Multi-hop RAG — iterative retrieval
- Optimizing the RAG pipeline with MIPROv2
- Evaluating with SemanticF1 and CompleteAndGrounded

**Runtime:** ~60 minutes

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/03_rag_pipeline.ipynb")

## 1. Setup — In-Memory Retriever

In [ ]:
import dspy

from src.config import get_dspy_lm
from src.datasets import generate_rag_contexts

lm = get_dspy_lm()
dspy.configure(lm=lm)

# Simple in-memory retriever for demo
documents = generate_rag_contexts(20)
corpus = {f"doc_{i}": d["context"] for i, d in enumerate(documents)}


class SimpleRetriever:
    def __init__(self, corpus, k=3):
        self.corpus = corpus
        self.k = k

    def __call__(self, query):
        # Simple keyword matching (replace with real vector DB in production)
        words = query.lower().split()
        scores = {
            doc_id: sum(1 for w in words if w in text.lower())
            for doc_id, text in self.corpus.items()
        }
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[: self.k]
        return dspy.Prediction(passages=[self.corpus[doc_id] for doc_id, _ in top if _ > 0])


retriever = SimpleRetriever(corpus)
print(f"✅ Retriever ready with {len(corpus)} documents")

## 2. Basic RAG Module

In [ ]:
class GenerateAnswer(dspy.Signature):
    """Answer questions based on context."""

    context: str = dspy.InputField()
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class BasicRAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()
        self.retrieve = retriever
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        passages = self.retrieve(question).passages
        context = "\n".join(passages)
        return self.generate(context=context, question=question)


rag = BasicRAG()
result = rag(question="What is the return policy?")
print("Question: What is the return policy?")
print(f"Answer: {result.answer}")

## 3. Adding `dspy.Assert` for Faithfulness

In [ ]:
class FaithfulRAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.retrieve = retriever
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        passages = self.retrieve(question).passages
        context = "\n".join(passages)
        result = self.generate(context=context, question=question)

        # Hard constraint: answer must be grounded in context
        dspy.Assert(
            any(word in context.lower() for word in result.answer.lower().split()[:3]),
            "Answer must be grounded in the provided context.",
        )

        return result


faithful_rag = FaithfulRAG()
try:
    result = faithful_rag(question="What is the return policy?")
    print(f"Answer: {result.answer}")
except AssertionError as e:
    print(f"Assertion failed: {e}")

## 4. Adding `dspy.Suggest` for Length Constraints

In [ ]:
class ConstrainedRAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.retrieve = retriever
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        passages = self.retrieve(question).passages
        context = "\n".join(passages)
        result = self.generate(context=context, question=question)

        # Soft constraint: prefer concise answers
        dspy.Suggest(
            len(result.answer.split()) <= 50, "Answer should be under 50 words for conciseness."
        )

        return result


constrained_rag = ConstrainedRAG()
result = constrained_rag(question="What is the return policy?")
print(f"Answer ({len(result.answer.split())} words): {result.answer}")

## 5. Multi-Hop RAG — Iterative Retrieval

In [ ]:
class GenerateSearchQuery(dspy.Signature):
    """Generate a search query based on context and question."""

    context: str = dspy.InputField()
    question: str = dspy.InputField()
    search_query: str = dspy.OutputField()


class MultiHopRAG(dspy.Module):
    def __init__(self, num_hops=2):
        super().__init__()
        self.retrieve = retriever
        self.generate_query = dspy.ChainOfThought(GenerateSearchQuery)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)
        self.num_hops = num_hops

    def forward(self, question):
        context = []
        for _ in range(self.num_hops):
            query = self.generate_query(context="\n".join(context), question=question).search_query
            passages = self.retrieve(query).passages
            context.extend(passages)

        return self.generate_answer(context="\n".join(context), question=question)


multi_hop = MultiHopRAG(num_hops=2)
result = multi_hop(question="What is the return policy?")
print(f"Multi-hop answer: {result.answer}")

## 6. Optimizing with MIPROv2

In [ ]:
from dspy.evaluate import Evaluate
from dspy.teleprompt import MIPROv2

# Prepare eval data
rag_data = generate_rag_contexts(20)
eval_examples = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question")
    for d in rag_data[:10]
]


def rag_metric(example, prediction, trace=None):
    return 1.0 if example.answer.lower() in prediction.answer.lower() else 0.0


# Optimize multi-hop RAG
mipro = MIPROv2(metric=rag_metric, num_candidates=3)
optimized_rag = mipro.compile(
    MultiHopRAG(num_hops=2),
    trainset=eval_examples[:7],
    num_trials=5,  # Reduced for cost
    valset=eval_examples[7:],
)

evaluator = Evaluate(devset=eval_examples[7:], metric=rag_metric, num_threads=2)
score = evaluator(optimized_rag)
print(f"\nOptimized RAG score: {score:.2f}")

## 7. Evaluating with SemanticF1 and CompleteAndGrounded

In [ ]:
# Note: These metrics require specific setup
# SemanticF1: semantic similarity between prediction and gold
# CompleteAndGrounded: checks coverage and faithfulness

print("Built-in RAG metrics:")
print("  • SemanticF1: Semantic similarity score")
print("  • CompleteAndGrounded: Coverage + faithfulness")
print("\nFor production, combine multiple metrics:")
print("  • Exact match for easy questions")
print("  • SemanticF1 for paraphrased answers")
print("  • CompleteAndGrounded for RAG faithfulness")

## 8. Exercise: Add Contradiction-Detection Assertion

Create an assertion that detects if the answer contradicts the retrieved context.

In [ ]:
# YOUR TURN: Contradiction detection

# class ContradictionAwareRAG(dspy.Module):
#     def forward(self, question):
#         ...
#         # Assert no contradiction
#         dspy.Assert(
#             not self.contradicts(context, result.answer),
#             "Answer contradicts the retrieved context."
#         )
#         return result

#     def contradicts(self, context, answer):
#         # Your contradiction detection logic
#         pass

---

**Next:** [04_react_agent.ipynb](04_react_agent.ipynb) — ReAct agents with real tools